In [4]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
import multiprocessing as mp

mp.set_start_method("fork", force=True)
df = pd.read_csv('../data/recipes_ready_for_es.csv')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

def parallelize_dataframe(df_col, func, n_cores=4):
    with mp.get_context("fork").Pool(processes=n_cores) as pool:
        return pool.map(func, df_col)

print("Parallel Processing...")
df['clean_content'] = parallelize_dataframe(
    (df['Name'] + " " + df['RecipeCategory'] + " " + df['Keywords']).fillna(""),
    clean_text
)
print("Preprocessing")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/chefthanathip/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Parallel Processing...
Preprocessing


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pickle

print(" TF-IDF...")
tfidf = TfidfVectorizer(max_features=10000)
tfidf_matrix = tfidf.fit_transform(df['clean_content'])

print(" SVD ...")
svd = TruncatedSVD(n_components=100, random_state=42)
latent_matrix = svd.fit_transform(tfidf_matrix)

with open('../models/tfidf_model.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('../models/svd_model.pkl', 'wb') as f: pickle.dump(svd, f)

recipe_features = pd.DataFrame(latent_matrix, index=df['RecipeId'].astype(str))
with open('../models/recipe_features.pkl', 'wb') as f: pickle.dump(recipe_features, f)

print(" Latent Features Success!")

 TF-IDF...
 SVD ...
 Latent Features Success!


In [6]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split


print(" Learning to Rank...")

numeric_features = df[['Calories', 'FatContent', 'ProteinContent', 'TotalTimeMins']].fillna(0)
X = np.hstack([latent_matrix, numeric_features.values])
y = df['AggregatedRating'].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


ranker = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=100,
    learning_rate=0.1
)

print("train LightGBM...")
ranker.fit(X_train, y_train)

with open('../models/lgbm_model.pkl', 'wb') as f: pickle.dump(ranker, f)
print("train LightGBM Ranker success")

 Learning to Rank...
train LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068033 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26520
[LightGBM] [Info] Number of data points in the train set: 418013, number of used features: 104
[LightGBM] [Info] Start training from score 2.386916
train LightGBM Ranker success


In [7]:
import optuna

def objective(trial):
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
    }

    gbm = lgb.train(param, lgb.Dataset(X_train, label=y_train))
    preds = gbm.predict(X_test)
    rmse = np.sqrt(((preds - y_test) ** 2).mean())
    return rmse

print(" Optuna ...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

print(f"Best: {study.best_params}")

/opt/anaconda3/envs/SE_481/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-27 09:58:50,080] A new study created in memory with name: no-name-7b0ede7c-3aac-4b1e-82aa-032601adb56d


 Optuna ...


[I 2026-03-27 09:59:04,317] Trial 0 finished with value: 2.2968198107843696 and parameters: {'learning_rate': 0.03154695174846775, 'num_leaves': 147, 'feature_fraction': 0.4727375724493424}. Best is trial 0 with value: 2.2968198107843696.
[I 2026-03-27 09:59:08,688] Trial 1 finished with value: 2.2941919953010053 and parameters: {'learning_rate': 0.21064857348010368, 'num_leaves': 106, 'feature_fraction': 0.5412185203898188}. Best is trial 1 with value: 2.2941919953010053.
[I 2026-03-27 09:59:17,803] Trial 2 finished with value: 2.2873978230546763 and parameters: {'learning_rate': 0.06886110984048772, 'num_leaves': 144, 'feature_fraction': 0.7397515008157588}. Best is trial 2 with value: 2.2873978230546763.
[I 2026-03-27 09:59:25,432] Trial 3 finished with value: 2.2990573171559143 and parameters: {'learning_rate': 0.2596770517263008, 'num_leaves': 93, 'feature_fraction': 0.6827034055972576}. Best is trial 2 with value: 2.2873978230546763.
[I 2026-03-27 09:59:31,242] Trial 4 finished w

Best: {'learning_rate': 0.06886110984048772, 'num_leaves': 144, 'feature_fraction': 0.7397515008157588}


In [8]:
import lightgbm as lgb
import pickle

best_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'learning_rate': 0.06886110984048772,
    'num_leaves': 144,
    'feature_fraction': 0.7397515008157588
}

final_model = lgb.LGBMRegressor(**best_params)

print(" Final Model...")
final_model.fit(X_train, y_train)

with open('../models/lgbm_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("Final model success")

 Final Model...
Final model success


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_pred = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"📊  (Regression Metrics):")
print(f"🔹 RMSE: {rmse:.4f} ")
print(f"🔹 MAE: {mae:.4f} ")


NameError: name 'final_model' is not defined

In [10]:
def test_recommendation(user_pref_text, top_k=5):
    user_tfidf = tfidf.transform([user_pref_text])
    user_dna = svd.transform(user_tfidf)

    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(user_dna, latent_matrix)[0]
    top_indices = sims.argsort()[-100:][::-1]


    candidate_dna = latent_matrix[top_indices]
    candidate_numeric = numeric_features.iloc[top_indices].values
    X_rank = np.hstack([candidate_dna, candidate_numeric])

    preds = final_model.predict(X_rank)

    results = df.iloc[top_indices].copy()
    results['predicted_rating'] = preds

    return results.sort_values('predicted_rating', ascending=False).head(top_k)[['Name', 'RecipeCategory', 'predicted_rating']]

print("🧪 ทดสอบ: User ชอบ 'Healthy, Salad, Low Carb'")
display(test_recommendation("Healthy, Salad, Low Carb"))

print("\n🧪 ทดสอบ: User ชอบ 'Pork, Egg, Chess'")
display(test_recommendation("Sweet, Dessert, Chocolate cake"))

🧪 ทดสอบ: User ชอบ 'Healthy, Salad, Low Carb'


/opt/anaconda3/envs/SE_481/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Name,RecipeCategory,predicted_rating
39668,Norma's Excellent Cranberry Salad,Low Cholesterol,2.476555
47959,Fat-Free Italian-Balsamic Salad Dressing,Salad Dressings,2.434997
217994,Hot Bacon Dressing (For Spinach-Bacon Salad),Salad Dressings,2.376824
35110,No-Oil Salad Dressing #2,Salad Dressings,2.358207
41919,Italian Salad Dressing Mix,Salad Dressings,2.341515



🧪 ทดสอบ: User ชอบ 'Sweet, Dessert, Chocolate cake'


/opt/anaconda3/envs/SE_481/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Name,RecipeCategory,predicted_rating
124088,Bittersweet Chocolate Cake,Dessert,2.873460
2400,Chocolate Espresso Fudge Cake,Dessert,2.835990
235407,Chocolate Blackout Cake,Dessert,2.829287
69444,Sauerkraut Chocolate Cake,Dessert,2.756370
292540,Real Chocolate Chocolate Cake With Ganache,Dessert,2.751483


In [6]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/recipes_ready_for_es.csv')
df['RecipeId'] = df['RecipeId'].astype(str)
df.set_index('RecipeId', inplace=True)

with open('../models/recipe_features.pkl', 'rb') as f:
    recipe_features = pickle.load(f)
with open('../models/lgbm_model.pkl', 'rb') as f:
    lgbm_model = pickle.load(f)


mock_bookmarks = [
    {'recipe_id': '38', 'rating': 5},
    {'recipe_id': '45', 'rating': 4},
    {'recipe_id': '122', 'rating': 1}
]

for b in mock_bookmarks:
    rid = b['recipe_id']
    if rid in df.index:
        name = df.loc[rid, 'Name']
        category = df.loc[rid, 'RecipeCategory']
        print(f" - 🏷️ [ID: {rid}] {name} (หมวด: {category}) | ให้คะแนน: {b['rating']} ดาว")
    else:
        print(f" - 🏷️ [ID: {rid}] (ไม่พบข้อมูลในตาราง)")

user_vector = np.zeros(100)
total_weight = 0

for b in mock_bookmarks:
    rid = b['recipe_id']
    if rid in recipe_features.index:
        recipe_dna = recipe_features.loc[rid].values
        weight = b['rating'] - 2.5
        user_vector += (recipe_dna * weight)
        total_weight += abs(weight)

if total_weight > 0:
    user_vector = user_vector / total_weight

sims = cosine_similarity(user_vector.reshape(1, -1), recipe_features.values)[0]
candidate_indices = sims.argsort()[-100:][::-1] # คัดมา 100 อันดับแรก
candidate_ids = recipe_features.index[candidate_indices]

candidate_dna = recipe_features.loc[candidate_ids].values
numeric_cols = ['Calories', 'FatContent', 'ProteinContent', 'TotalTimeMins']
candidate_numeric = df.loc[candidate_ids, numeric_cols].fillna(0).values

X_rank = np.hstack([candidate_dna, candidate_numeric])

predicted_ratings = lgbm_model.predict(X_rank)

results_df = pd.DataFrame({
    'RecipeId': candidate_ids,
    'Name': df.loc[candidate_ids, 'Name'].values,
    'Category': df.loc[candidate_ids, 'RecipeCategory'].values,
    'Ingredients': df.loc[candidate_ids, 'RecipeIngredientParts'].values,
    'Sim_Score': sims[candidate_indices],
    'Predicted_Rating': predicted_ratings
})

final_recs = results_df.sort_values('Predicted_Rating', ascending=False).head(5)

print("\n🌟 เมนูแนะนำสำหรับ User คนนี้ (ผ่านการจัดอันดับด้วย LightGBM):")
for i, row in final_recs.iterrows():
    print(f"🔸 [ID: {row['RecipeId']}] {row['Name']}")
    print(f"   🍲 หมวดหมู่: {row['Category']}")
    print(f"   🥗 ส่วนผสม: {row['Ingredients'][:100]}...")
    print(f"   🤖 AI คาดเดาว่าจะได้เรตติ้ง: {row['Predicted_Rating']:.2f} ดาว")
    print(f"   (ความคล้าย DNA พื้นฐาน: {row['Sim_Score']:.4f})\n")

กำลังโหลดข้อมูลฐานอาหาร...
กำลังโหลด Models สมองกล...

📌 ประวัติการบันทึก (Bookmarks) ของ User:
 - 🏷️ [ID: 38] Low-Fat Berry Blue Frozen Dessert (หมวด: Frozen Desserts) | ให้คะแนน: 5 ดาว
 - 🏷️ [ID: 45] Buttermilk Pie With Gingersnap Crumb Crust (หมวด: Pie) | ให้คะแนน: 4 ดาว
 - 🏷️ [ID: 122] Commissary Carrot Cake (หมวด: Dessert) | ให้คะแนน: 1 ดาว

🌟 เมนูแนะนำสำหรับ User คนนี้ (ผ่านการจัดอันดับด้วย LightGBM):
🔸 [ID: 52548] Lemon Frozen Yogurt
   🍲 หมวดหมู่: Frozen Desserts
   🥗 ส่วนผสม: low-fat vanilla yogurt, lemon juice, sugar, light corn syrup, lemon, zest of, vanilla extract...
   🤖 AI คาดเดาว่าจะได้เรตติ้ง: 2.61 ดาว
   (ความคล้าย DNA พื้นฐาน: 0.7113)

🔸 [ID: 142165] Frozen Orange Swirl Pie
   🍲 หมวดหมู่: Pie
   🥗 ส่วนผสม: vanilla ice cream...
   🤖 AI คาดเดาว่าจะได้เรตติ้ง: 2.56 ดาว
   (ความคล้าย DNA พื้นฐาน: 0.7232)

🔸 [ID: 76489] Frozen Pistachio Cookie Dessert
   🍲 หมวดหมู่: Frozen Desserts
   🥗 ส่วนผสม: margarine, skim milk, instant pistachio pudding mix, Cool Whip Lite...
   🤖 A

/opt/anaconda3/envs/SE_481/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
